# Notebook 09: Streamlit Application Validation

## Objective

Validate that the Streamlit application correctly uses the finalized deployment pipeline without retraining or modifying the model.

This notebook verifies:

- the finalized preprocessor and Tuned XGBoost model
- the 43 raw predictors
- the 179 transformed features
- feature order consistency
- the two finalized screening thresholds
- direct-entry and CSV prediction consistency
- record-level explanation consistency
- invalid-input handling
- downloadable result structures
- readable prediction-factor downloads
- direct-entry confirmation controls
- the complete eight-page application structure
- the Application Validation dashboard and saved evidence files


### Deployment asset validation

In [ ]:
#Load and validate final deployment assets

from pathlib import Path
import json
import sys

import joblib
import pandas as pd

# 1. Locate the project root safely
def find_project_root(start_path: Path) -> Path:
    """
    Search the current directory and its parents for the project root.

    The project root must contain:
    - models/
    - artifacts/
    - prediction_service.py
    - custom_transformers.py
    """

    required_items = [
        "models",
        "artifacts",
        "prediction_service.py",
        "custom_transformers.py",
    ]

    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if all((candidate / item).exists() for item in required_items):
            return candidate

    raise FileNotFoundError(
        "Could not locate the hospital-readmission-project root. "
        "Run this notebook from inside the project folder."
    )


CURRENT_LOCATION = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_LOCATION)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook location : {CURRENT_LOCATION}")
print(f"Project root      : {PROJECT_ROOT}")


# 2. Import the custom transformer before loading joblib files

from custom_transformers import RareCategoryGrouper  # noqa: E402, F401

# 3. Define finalized deployment-asset paths
PREPROCESSOR_PATH = (
    PROJECT_ROOT
    / "models"
    / "final_preprocessor.joblib"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "final_xgboost_model.joblib"
)

INPUT_SCHEMA_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "streamlit_input_schema.json"
)

DEPLOYMENT_CONFIG_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "final_deployment_config.json"
)

required_paths = {
    "Final preprocessor": PREPROCESSOR_PATH,
    "Final XGBoost model": MODEL_PATH,
    "Streamlit input schema": INPUT_SCHEMA_PATH,
    "Deployment configuration": DEPLOYMENT_CONFIG_PATH,
}

# 4. Confirm every required file exists
missing_assets = [
    f"{asset_name}: {asset_path}"
    for asset_name, asset_path in required_paths.items()
    if not asset_path.exists()
]

if missing_assets:
    raise FileNotFoundError(
        "The following deployment assets are missing:\n"
        + "\n".join(missing_assets)
    )

print("\nRequired deployment assets found:")

for asset_name, asset_path in required_paths.items():
    print(f"  [FOUND] {asset_name}: {asset_path.relative_to(PROJECT_ROOT)}")

# 5. Load JSON configuration files
with open(INPUT_SCHEMA_PATH, "r", encoding="utf-8") as schema_file:
    input_schema = json.load(schema_file)

with open(
    DEPLOYMENT_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as config_file:
    deployment_config = json.load(config_file)

# 6. Load final preprocessing and modeling objects
final_preprocessor = joblib.load(PREPROCESSOR_PATH)
final_model = joblib.load(MODEL_PATH)

# 7. Extract validation values
schema_feature_order = input_schema["feature_order"]

schema_raw_feature_count = int(input_schema["feature_count"])
schema_numeric_count = int(input_schema["numeric_feature_count"])
schema_categorical_count = int(
    input_schema["categorical_feature_count"]
)

preprocessor_raw_feature_count = int(
    final_preprocessor.n_features_in_
)

model_transformed_feature_count = int(
    final_model.n_features_in_
)

transformed_feature_names = (
    final_preprocessor
    .get_feature_names_out()
    .astype(str)
    .tolist()
)

preprocessor_feature_order = (
    final_preprocessor
    .feature_names_in_
    .astype(str)
    .tolist()
)

# 8. Validate finalized deployment requirements
validation_checks = {
    "Input schema contains 43 predictors": (
        schema_raw_feature_count == 43
    ),
    "Feature-order list contains 43 predictors": (
        len(schema_feature_order) == 43
    ),
    "Schema contains 8 numeric predictors": (
        schema_numeric_count == 8
    ),
    "Schema contains 35 categorical predictors": (
        schema_categorical_count == 35
    ),
    "Numeric and categorical counts total 43": (
        schema_numeric_count + schema_categorical_count == 43
    ),
    "Preprocessor expects 43 raw predictors": (
        preprocessor_raw_feature_count == 43
    ),
    "Preprocessor feature order matches schema": (
        preprocessor_feature_order == schema_feature_order
    ),
    "Preprocessor produces 179 transformed features": (
        len(transformed_feature_names) == 179
    ),
    "XGBoost model expects 179 transformed features": (
        model_transformed_feature_count == 179
    ),
    "Final model supports predict_proba": (
        hasattr(final_model, "predict_proba")
    ),
    "Main threshold equals 0.50": (
        float(input_schema["main_threshold"]) == 0.50
    ),
    "Recall-focused threshold equals 0.45": (
        float(input_schema["recall_focused_threshold"]) == 0.45
    ),
}

validation_table = pd.DataFrame(
    [
        {
            "Validation Check": check_name,
            "Result": "PASSED" if passed else "FAILED",
        }
        for check_name, passed in validation_checks.items()
    ]
)

display(validation_table)

# 9. Stop immediately if any validation failed
failed_checks = [
    check_name
    for check_name, passed in validation_checks.items()
    if not passed
]

if failed_checks:
    raise AssertionError(
        "Deployment-asset validation failed:\n"
        + "\n".join(f"- {check}" for check in failed_checks)
    )

# 10. Display final deployment summary
deployment_summary = pd.DataFrame(
    {
        "Deployment Property": [
            "Final model type",
            "Preprocessor type",
            "Raw predictors",
            "Numeric predictors",
            "Categorical predictors",
            "Transformed features",
            "Main threshold",
            "Recall-focused threshold",
            "Prediction target",
            "Positive class",
        ],
        "Validated Value": [
            type(final_model).__name__,
            type(final_preprocessor).__name__,
            schema_raw_feature_count,
            schema_numeric_count,
            schema_categorical_count,
            len(transformed_feature_names),
            input_schema["main_threshold"],
            input_schema["recall_focused_threshold"],
            input_schema["prediction_target"],
            input_schema["positive_class"],
        ],
    }
)

display(deployment_summary)

print("\n" + "=" * 68)
print("STEP 1 RESULT: PASSED")
print("All finalized deployment assets loaded and validated successfully.")
print("=" * 68)

## Step 2: Guided Form Configuration Validation

This step verifies that the direct-entry Streamlit form:

- contains all 43 required predictors
- contains each predictor exactly once
- has no missing or unexpected fields
- organizes fields into the intended four entry sections
- uses valid numeric defaults and ranges
- uses valid categorical defaults and options
- contains readable labels for every predictor
- correctly separates the primary diabetes fields from the additional medication fields

In [ ]:
from collections import Counter

from ui.form_config import (
    ADDITIONAL_MEDICATION_FIELDS,
    FIELD_LABELS,
    FORM_STEPS,
    PRIMARY_DIABETES_FIELDS,
    get_all_configured_features,
    get_feature_label,
    validate_form_configuration,
)

# 1. Read the expected schema structure
expected_feature_order = input_schema["feature_order"]
numeric_schema = input_schema["numeric_features"]
categorical_schema = input_schema["categorical_features"]

configured_features = get_all_configured_features()
configured_feature_counts = Counter(configured_features)

expected_section_counts = {
    "patient_profile": 3,
    "hospital_encounter": 12,
    "previous_healthcare_use": 3,
    "diabetes_management": 25,
}

# 2. Create a section-level summary
section_summary_rows = []

for step_number, section in enumerate(FORM_STEPS, start=1):
    section_fields = section["fields"]

    section_summary_rows.append(
        {
            "Step": step_number,
            "Section Key": section["key"],
            "Section Title": section["title"],
            "Configured Fields": len(section_fields),
            "Expected Fields": expected_section_counts.get(
                section["key"]
            ),
            "Field Count Match": (
                len(section_fields)
                == expected_section_counts.get(section["key"])
            ),
        }
    )

form_section_summary = pd.DataFrame(section_summary_rows)

display(form_section_summary)

# 3. Compare the form configuration with the model schema
configuration_comparison = validate_form_configuration(
    expected_feature_order
)

missing_features = configuration_comparison["missing_features"]
unexpected_features = configuration_comparison[
    "unexpected_features"
]
duplicate_features = configuration_comparison[
    "duplicate_features"
]

configured_feature_set = set(configured_features)
expected_feature_set = set(expected_feature_order)

# 4. Validate numeric defaults and ranges
invalid_numeric_defaults = []

for feature, settings in numeric_schema.items():
    minimum = settings["minimum"]
    maximum = settings["maximum"]
    default = settings["default"]
    step = settings["step"]

    if not minimum <= default <= maximum:
        invalid_numeric_defaults.append(
            {
                "Feature": feature,
                "Problem": "Default is outside the allowed range",
                "Minimum": minimum,
                "Default": default,
                "Maximum": maximum,
            }
        )

    if step <= 0:
        invalid_numeric_defaults.append(
            {
                "Feature": feature,
                "Problem": "Step must be greater than zero",
                "Minimum": minimum,
                "Default": default,
                "Maximum": maximum,
            }
        )

# 5. Validate categorical defaults and options
invalid_categorical_defaults = []

for feature, settings in categorical_schema.items():
    options = settings["options"]
    default = settings["default"]

    if not options:
        invalid_categorical_defaults.append(
            {
                "Feature": feature,
                "Problem": "No categorical options are available",
                "Default": default,
            }
        )

    elif default not in options:
        invalid_categorical_defaults.append(
            {
                "Feature": feature,
                "Problem": "Default is not included in options",
                "Default": default,
            }
        )

# 6. Validate readable labels
missing_readable_labels = [
    feature
    for feature in expected_feature_order
    if feature not in FIELD_LABELS
    or not str(FIELD_LABELS[feature]).strip()
]

readable_label_rows = [
    {
        "Internal Feature": feature,
        "User-Facing Label": get_feature_label(feature),
        "Form Section": next(
            section["title"]
            for section in FORM_STEPS
            if feature in section["fields"]
        ),
    }
    for feature in configured_features
]

readable_label_table = pd.DataFrame(readable_label_rows)

display(readable_label_table)

# 7. Validate diabetes-field organization
diabetes_section = next(
    section
    for section in FORM_STEPS
    if section["key"] == "diabetes_management"
)

diabetes_section_fields = set(diabetes_section["fields"])
primary_diabetes_set = set(PRIMARY_DIABETES_FIELDS)
additional_medication_set = set(
    ADDITIONAL_MEDICATION_FIELDS
)

diabetes_overlap = sorted(
    primary_diabetes_set.intersection(
        additional_medication_set
    )
)

combined_diabetes_fields = (
    primary_diabetes_set
    | additional_medication_set
)

missing_from_diabetes_groups = sorted(
    diabetes_section_fields
    - combined_diabetes_fields
)

unexpected_in_diabetes_groups = sorted(
    combined_diabetes_fields
    - diabetes_section_fields
)

# 8. Run all guided-form validation checks
form_validation_checks = {
    "Form contains four data-entry sections": (
        len(FORM_STEPS) == 4
    ),
    "All section keys are unique": (
        len({section["key"] for section in FORM_STEPS})
        == len(FORM_STEPS)
    ),
    "All section titles are unique": (
        len({section["title"] for section in FORM_STEPS})
        == len(FORM_STEPS)
    ),
    "Every section has the expected field count": (
        form_section_summary["Field Count Match"].all()
    ),
    "Form contains exactly 43 configured fields": (
        len(configured_features) == 43
    ),
    "Form contains 43 unique fields": (
        len(configured_feature_set) == 43
    ),
    "Schema contains 43 unique fields": (
        len(expected_feature_set) == 43
    ),
    "No required predictors are missing": (
        len(missing_features) == 0
    ),
    "No unexpected predictors are configured": (
        len(unexpected_features) == 0
    ),
    "No predictor appears more than once": (
        len(duplicate_features) == 0
    ),
    "Configured field set matches schema field set": (
        configured_feature_set == expected_feature_set
    ),
    "All numeric defaults and ranges are valid": (
        len(invalid_numeric_defaults) == 0
    ),
    "All categorical defaults and options are valid": (
        len(invalid_categorical_defaults) == 0
    ),
    "Every predictor has a readable label": (
        len(missing_readable_labels) == 0
    ),
    "Primary diabetes section contains five fields": (
        len(PRIMARY_DIABETES_FIELDS) == 5
    ),
    "Additional medication section contains twenty fields": (
        len(ADDITIONAL_MEDICATION_FIELDS) == 20
    ),
    "Primary and additional diabetes groups do not overlap": (
        len(diabetes_overlap) == 0
    ),
    "Diabetes groups cover the complete diabetes section": (
        len(missing_from_diabetes_groups) == 0
        and len(unexpected_in_diabetes_groups) == 0
    ),
}

form_validation_table = pd.DataFrame(
    [
        {
            "Validation Check": check_name,
            "Result": "PASSED" if passed else "FAILED",
        }
        for check_name, passed in form_validation_checks.items()
    ]
)

display(form_validation_table)

# 9. Report detailed errors when present
form_validation_details = {
    "Missing predictors": missing_features,
    "Unexpected predictors": unexpected_features,
    "Duplicate predictors": duplicate_features,
    "Missing readable labels": missing_readable_labels,
    "Invalid numeric defaults": invalid_numeric_defaults,
    "Invalid categorical defaults": invalid_categorical_defaults,
    "Overlapping diabetes fields": diabetes_overlap,
    "Diabetes fields not assigned to a subgroup": (
        missing_from_diabetes_groups
    ),
    "Unexpected diabetes subgroup fields": (
        unexpected_in_diabetes_groups
    ),
}

validation_issue_rows = []

for issue_name, issue_values in form_validation_details.items():
    if issue_values:
        validation_issue_rows.append(
            {
                "Issue Type": issue_name,
                "Details": str(issue_values),
            }
        )

if validation_issue_rows:
    validation_issue_table = pd.DataFrame(
        validation_issue_rows
    )
    display(validation_issue_table)
else:
    print("No guided-form configuration issues were found.")

# 10. Stop immediately if any check failed
failed_form_checks = [
    check_name
    for check_name, passed in form_validation_checks.items()
    if not passed
]

if failed_form_checks:
    raise AssertionError(
        "Guided-form validation failed:\n"
        + "\n".join(
            f"- {check}"
            for check in failed_form_checks
        )
    )

# 11. Preserve results for the final notebook summary
step_2_form_section_summary = form_section_summary.copy()
step_2_form_validation_table = form_validation_table.copy()
step_2_readable_label_table = readable_label_table.copy()


print("\n" + "=" * 68)
print("STEP 2 RESULT: PASSED")
print("The guided form contains all 43 required predictors exactly once.")
print("All defaults, ranges, options, labels, and sections are valid.")
print("=" * 68)

## Step 3: Direct-Entry and CSV Prediction Parity

The Streamlit application supports both guided single-record entry and CSV upload.

This step verifies that both input methods use the same finalized:

- 43-predictor schema
- preprocessing pipeline
- Tuned XGBoost model
- 0.50 standard-review cutoff
- 0.45 additional-screening cutoff
- record-level explanation method

The predictions and explanations must match when identical input values are used.

In [ ]:
import numpy as np

from prediction_service import (
    explain_readmission_batch,
    predict_readmission,
    predict_readmission_batch,
    validate_batch_input,
)

# 1. Locate the permanent synthetic sample file
SAMPLE_INPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "sample_patient_input.csv"
)

if not SAMPLE_INPUT_PATH.exists():
    raise FileNotFoundError(
        "The synthetic sample input file could not be found:\n"
        f"{SAMPLE_INPUT_PATH}"
    )

sample_input_raw = pd.read_csv(SAMPLE_INPUT_PATH)

print(f"Sample input path : {SAMPLE_INPUT_PATH}")
print(f"Raw sample shape  : {sample_input_raw.shape}")

# 2. Validate the sample using the deployed application logic
sample_input_validated = validate_batch_input(
    sample_input_raw
)

expected_feature_order = input_schema["feature_order"]

sample_validation_checks = {
    "Sample contains at least one record": (
        len(sample_input_validated) >= 1
    ),
    "Sample contains exactly 43 predictors": (
        sample_input_validated.shape[1] == 43
    ),
    "Sample column order matches deployment schema": (
        sample_input_validated.columns.tolist()
        == expected_feature_order
    ),
}

sample_validation_table = pd.DataFrame(
    [
        {
            "Validation Check": check_name,
            "Result": "PASSED" if passed else "FAILED",
        }
        for check_name, passed
        in sample_validation_checks.items()
    ]
)

display(sample_validation_table)

failed_sample_checks = [
    check_name
    for check_name, passed
    in sample_validation_checks.items()
    if not passed
]

if failed_sample_checks:
    raise AssertionError(
        "Synthetic sample validation failed:\n"
        + "\n".join(
            f"- {check}"
            for check in failed_sample_checks
        )
    )

# 3. Generate CSV batch predictions
batch_prediction_results = predict_readmission_batch(
    sample_input_validated
)

expected_batch_rows = len(sample_input_validated)

if len(batch_prediction_results) != expected_batch_rows:
    raise AssertionError(
        "The number of batch predictions does not match "
        "the number of validated sample records."
    )

display(batch_prediction_results)

# 4. Generate batch record-level explanations
batch_explanation_results = explain_readmission_batch(
    sample_input_validated,
    top_n=5,
)

expected_explanation_rows = expected_batch_rows * 10

if len(batch_explanation_results) != expected_explanation_rows:
    raise AssertionError(
        "Expected ten explanation rows per record "
        f"but received {len(batch_explanation_results)} rows."
    )

# 5. Helper functions for expected classifications
def expected_main_classification(
    prediction_result: dict,
) -> str:
    """
    Convert the direct-entry prediction output to the same
    technical classification used by the batch service.
    """

    if prediction_result["main_threshold_prediction"] == 1:
        return "Flagged at Main Threshold"

    return "Not Flagged at Main Threshold"


def expected_screening_classification(
    prediction_result: dict,
) -> str:
    """
    Convert the direct-entry prediction output to the same
    technical classification used by the batch service.
    """

    if (
        prediction_result["recall_focused_prediction"]
        == 1
    ):
        return "Flagged for Screening"

    return "Not Flagged"

# 6. Compare direct-entry and CSV results record by record
PROBABILITY_TOLERANCE = 1e-12
EXPLANATION_PERCENTAGE_TOLERANCE = 0.001

parity_rows = []
explanation_comparison_rows = []

explanation_comparison_columns = [
    "Direction",
    "Factor Rank",
    "Feature",
    "Patient Value",
    "Original Feature",
]

for record_index in range(expected_batch_rows):
    record_number = record_index + 1

    # Build the same dictionary used by direct-entry prediction.
    direct_input_values = {
        feature: sample_input_validated.iloc[
            record_index
        ][feature]
        for feature in expected_feature_order
    }

    # Generate direct-entry prediction.
    direct_prediction = predict_readmission(
        direct_input_values
    )

    # Retrieve the corresponding CSV batch result.
    batch_prediction = batch_prediction_results.iloc[
        record_index
    ]

    direct_probability = float(
        direct_prediction["probability"]
    )

    batch_probability = float(
        batch_prediction["Readmission Probability"]
    )

    probability_difference = abs(
        direct_probability - batch_probability
    )

    probability_match = (
        probability_difference
        <= PROBABILITY_TOLERANCE
    )

    main_classification_match = (
        batch_prediction["Main Classification"]
        == expected_main_classification(
            direct_prediction
        )
    )

    screening_classification_match = (
        batch_prediction[
            "Recall-Focused Classification"
        ]
        == expected_screening_classification(
            direct_prediction
        )
    )

    # Generate an explanation for the same record by itself.
    direct_record_dataframe = pd.DataFrame(
        [direct_input_values],
        columns=expected_feature_order,
    )

    direct_explanation = explain_readmission_batch(
        direct_record_dataframe,
        top_n=5,
    )

    # Retrieve the matching record from the batch explanation.
    batch_record_explanation = (
        batch_explanation_results[
            batch_explanation_results["Record Number"]
            == record_number
        ]
        .copy()
        .sort_values(
            ["Direction", "Factor Rank"]
        )
        .reset_index(drop=True)
    )

    direct_explanation_comparison = (
        direct_explanation[
            explanation_comparison_columns
        ]
        .astype(str)
        .sort_values(
            ["Direction", "Factor Rank"]
        )
        .reset_index(drop=True)
    )

    batch_explanation_comparison = (
        batch_record_explanation[
            explanation_comparison_columns
        ]
        .astype(str)
        .sort_values(
            ["Direction", "Factor Rank"]
        )
        .reset_index(drop=True)
    )

    explanation_factors_match = (
        direct_explanation_comparison.equals(
            batch_explanation_comparison
        )
    )

    explanation_probability = float(
        direct_explanation[
            "Readmission Probability (%)"
        ].iloc[0]
    )

    direct_probability_percentage = (
        direct_probability * 100
    )

    explanation_probability_difference = abs(
        explanation_probability
        - direct_probability_percentage
    )

    explanation_probability_match = np.isclose(
        explanation_probability,
        direct_probability_percentage,
        atol=EXPLANATION_PERCENTAGE_TOLERANCE,
        rtol=0,
    )

    parity_rows.append(
        {
            "Record Number": record_number,
            "Direct Probability (%)": (
                direct_probability_percentage
            ),
            "CSV Probability (%)": (
                batch_probability * 100
            ),
            "Probability Difference": (
                probability_difference
            ),
            "Probability Match": probability_match,
            "Standard Review Match": (
                main_classification_match
            ),
            "Additional Screening Match": (
                screening_classification_match
            ),
            "Explanation Factors Match": (
                explanation_factors_match
            ),
            "Explanation Probability Match": (
                explanation_probability_match
            ),
        }
    )

    explanation_comparison_rows.append(
        {
            "Record Number": record_number,
            "Model Probability (%)": (
                direct_probability_percentage
            ),
            "Explanation Probability (%)": (
                explanation_probability
            ),
            "Difference in Percentage Points": (
                explanation_probability_difference
            ),
            "Allowed Tolerance": (
                EXPLANATION_PERCENTAGE_TOLERANCE
            ),
            "Result": (
                "PASSED"
                if explanation_probability_match
                else "FAILED"
            ),
        }
    )

# 7. Display record-level parity results
direct_csv_parity_table = pd.DataFrame(
    parity_rows
)

explanation_probability_table = pd.DataFrame(
    explanation_comparison_rows
)

display(
    direct_csv_parity_table.style.format(
        {
            "Direct Probability (%)": "{:.8f}",
            "CSV Probability (%)": "{:.8f}",
            "Probability Difference": "{:.12f}",
        }
    )
)

display(
    explanation_probability_table.style.format(
        {
            "Model Probability (%)": "{:.8f}",
            "Explanation Probability (%)": "{:.8f}",
            "Difference in Percentage Points": (
                "{:.8f}"
            ),
            "Allowed Tolerance": "{:.3f}",
        }
    )
)

# 8. Create final Step 3 validation checks
step_3_validation_checks = {
    "Batch prediction returns one result per record": (
        len(batch_prediction_results)
        == expected_batch_rows
    ),
    "Each record receives ten explanation factors": (
        len(batch_explanation_results)
        == expected_explanation_rows
    ),
    "All direct and CSV probabilities match": (
        direct_csv_parity_table[
            "Probability Match"
        ].all()
    ),
    "All standard-review classifications match": (
        direct_csv_parity_table[
            "Standard Review Match"
        ].all()
    ),
    "All additional-screening classifications match": (
        direct_csv_parity_table[
            "Additional Screening Match"
        ].all()
    ),
    "All explanation factor rankings match": (
        direct_csv_parity_table[
            "Explanation Factors Match"
        ].all()
    ),
    "All explanation probabilities match model probabilities": (
        direct_csv_parity_table[
            "Explanation Probability Match"
        ].all()
    ),
}

step_3_validation_table = pd.DataFrame(
    [
        {
            "Validation Check": check_name,
            "Result": "PASSED" if passed else "FAILED",
        }
        for check_name, passed
        in step_3_validation_checks.items()
    ]
)

display(step_3_validation_table)

# 9. Stop immediately if any parity check failed
failed_parity_checks = [
    check_name
    for check_name, passed
    in step_3_validation_checks.items()
    if not passed
]

if failed_parity_checks:
    raise AssertionError(
        "Direct-entry and CSV parity validation failed:\n"
        + "\n".join(
            f"- {check}"
            for check in failed_parity_checks
        )
    )

# 10. Preserve results for later notebook output files
step_3_direct_csv_parity_table = (
    direct_csv_parity_table.copy()
)

step_3_explanation_probability_table = (
    explanation_probability_table.copy()
)

step_3_validation_summary = (
    step_3_validation_table.copy()
)


print("\n" + "=" * 72)
print("STEP 3 RESULT: PASSED")
print(
    "Direct-entry and CSV upload produce identical "
    "predictions, classifications, and explanation factors."
)
print("=" * 72)

## Step 4: Invalid-Input and Error-Handling Validation

This step verifies that the application rejects invalid inputs with clear
error messages before they reach the preprocessing pipeline or final model.

The tests cover:

- valid CSV records
- shuffled CSV columns
- accidental spaces around CSV column names
- non-DataFrame input
- empty CSV input
- duplicate columns
- missing required columns
- unexpected columns
- invalid numeric text
- missing numeric values
- numeric values below and above allowed ranges
- missing categorical values
- blank categorical values
- missing direct-entry predictors
- unexpected direct-entry predictors
- invalid direct-entry numeric values

In [ ]:
from typing import Callable

from prediction_service import (
    create_input_dataframe,
    validate_batch_input,
)

# 1. Prepare valid reference inputs
valid_batch_reference = sample_input_validated.copy()

valid_direct_reference = {
    feature: valid_batch_reference.iloc[0][feature]
    for feature in expected_feature_order
}

numeric_feature = next(
    iter(input_schema["numeric_features"])
)

categorical_feature = next(
    iter(input_schema["categorical_features"])
)

numeric_settings = input_schema[
    "numeric_features"
][numeric_feature]

numeric_minimum = numeric_settings["minimum"]
numeric_maximum = numeric_settings["maximum"]
numeric_step = numeric_settings["step"]

range_increment = (
    numeric_step
    if numeric_step > 0
    else 1
)

print(f"Numeric test feature     : {numeric_feature}")
print(f"Allowed numeric range    : {numeric_minimum} to {numeric_maximum}")
print(f"Categorical test feature : {categorical_feature}")

# 2. Helper functions for successful validation cases
def validate_batch_and_confirm_order(
    input_dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Validate a batch input and confirm that the returned columns
    exactly match the deployment schema order.
    """

    validated_dataframe = validate_batch_input(
        input_dataframe
    )

    if validated_dataframe.columns.tolist() != expected_feature_order:
        raise AssertionError(
            "Validated CSV columns do not match the deployment schema."
        )

    if validated_dataframe.shape[1] != 43:
        raise AssertionError(
            "Validated CSV does not contain exactly 43 predictors."
        )

    return validated_dataframe


def validate_direct_and_confirm_order(
    input_values: dict,
) -> pd.DataFrame:
    """
    Validate direct-entry values and confirm the resulting
    one-row DataFrame matches the deployment schema.
    """

    direct_dataframe = create_input_dataframe(
        input_values
    )

    if direct_dataframe.shape != (1, 43):
        raise AssertionError(
            "Direct-entry validation did not produce a 1 × 43 DataFrame."
        )

    if direct_dataframe.columns.tolist() != expected_feature_order:
        raise AssertionError(
            "Direct-entry columns do not match the deployment schema."
        )

    return direct_dataframe

# 3. Build successful CSV validation cases
whitespace_column_case = valid_batch_reference.copy()
whitespace_column_case.columns = [
    f"  {column}  "
    for column in whitespace_column_case.columns
]

shuffled_column_case = valid_batch_reference[
    list(reversed(expected_feature_order))
].copy()


# 4. Build invalid CSV validation cases

empty_dataframe_case = pd.DataFrame(
    columns=expected_feature_order
)

duplicate_column_case = valid_batch_reference.copy()
duplicate_column_names = (
    duplicate_column_case.columns.tolist()
)
duplicate_column_names[1] = duplicate_column_names[0]
duplicate_column_case.columns = duplicate_column_names

missing_column_case = valid_batch_reference.drop(
    columns=[expected_feature_order[0]]
)

unexpected_column_case = valid_batch_reference.copy()
unexpected_column_case[
    "unexpected_predictor"
] = "unexpected"

invalid_numeric_text_case = valid_batch_reference.copy()
invalid_numeric_text_case[numeric_feature] = (
    invalid_numeric_text_case[numeric_feature]
    .astype(object)
)
invalid_numeric_text_case.loc[
    invalid_numeric_text_case.index[0],
    numeric_feature,
] = "not_a_number"

missing_numeric_case = valid_batch_reference.copy()
missing_numeric_case[numeric_feature] = (
    missing_numeric_case[numeric_feature]
    .astype(object)
)
missing_numeric_case.loc[
    missing_numeric_case.index[0],
    numeric_feature,
] = np.nan

below_minimum_case = valid_batch_reference.copy()
below_minimum_case.loc[
    below_minimum_case.index[0],
    numeric_feature,
] = numeric_minimum - range_increment

above_maximum_case = valid_batch_reference.copy()
above_maximum_case.loc[
    above_maximum_case.index[0],
    numeric_feature,
] = numeric_maximum + range_increment

missing_categorical_case = valid_batch_reference.copy()
missing_categorical_case[categorical_feature] = (
    missing_categorical_case[categorical_feature]
    .astype(object)
)
missing_categorical_case.loc[
    missing_categorical_case.index[0],
    categorical_feature,
] = pd.NA

blank_categorical_case = valid_batch_reference.copy()
blank_categorical_case[categorical_feature] = (
    blank_categorical_case[categorical_feature]
    .astype(object)
)
blank_categorical_case.loc[
    blank_categorical_case.index[0],
    categorical_feature,
] = "   "


# 5. Build invalid direct-entry cases
missing_direct_feature_case = (
    valid_direct_reference.copy()
)
missing_direct_feature_case.pop(
    expected_feature_order[0]
)

unexpected_direct_feature_case = (
    valid_direct_reference.copy()
)
unexpected_direct_feature_case[
    "unexpected_predictor"
] = "unexpected"

invalid_direct_numeric_case = (
    valid_direct_reference.copy()
)
invalid_direct_numeric_case[
    numeric_feature
] = "not_a_number"

# 6. Define all validation test cases
validation_test_cases = [
    {
        "Test Name": "Valid CSV input",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_and_confirm_order(
            valid_batch_reference.copy()
        ),
        "Expected Outcome": "Accepted",
        "Expected Exception": None,
        "Expected Message": None,
    },
    {
        "Test Name": "CSV columns with surrounding spaces",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_and_confirm_order(
            whitespace_column_case.copy()
        ),
        "Expected Outcome": "Accepted",
        "Expected Exception": None,
        "Expected Message": None,
    },
    {
        "Test Name": "CSV columns in shuffled order",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_and_confirm_order(
            shuffled_column_case.copy()
        ),
        "Expected Outcome": "Accepted",
        "Expected Exception": None,
        "Expected Message": None,
    },
    {
        "Test Name": "Valid direct-entry dictionary",
        "Input Method": "Direct Entry",
        "Callable": lambda: validate_direct_and_confirm_order(
            valid_direct_reference.copy()
        ),
        "Expected Outcome": "Accepted",
        "Expected Exception": None,
        "Expected Message": None,
    },
    {
        "Test Name": "Non-DataFrame CSV input",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            ["not", "a", "dataframe"]
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": TypeError,
        "Expected Message": (
            "uploaded input must be a pandas DataFrame"
        ),
    },
    {
        "Test Name": "Empty CSV input",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            empty_dataframe_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "does not contain any patient records"
        ),
    },
    {
        "Test Name": "Duplicate CSV column",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            duplicate_column_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": "Duplicate columns found",
    },
    {
        "Test Name": "Missing required CSV column",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            missing_column_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "CSV is missing required columns"
        ),
    },
    {
        "Test Name": "Unexpected CSV column",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            unexpected_column_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "CSV contains unexpected columns"
        ),
    },
    {
        "Test Name": "Invalid numeric text in CSV",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            invalid_numeric_text_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "Invalid or missing numeric value"
        ),
    },
    {
        "Test Name": "Missing numeric value in CSV",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            missing_numeric_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "Invalid or missing numeric value"
        ),
    },
    {
        "Test Name": "Numeric value below minimum",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            below_minimum_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": "must be between",
    },
    {
        "Test Name": "Numeric value above maximum",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            above_maximum_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": "must be between",
    },
    {
        "Test Name": "Missing categorical value in CSV",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            missing_categorical_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "Missing categorical value"
        ),
    },
    {
        "Test Name": "Blank categorical value in CSV",
        "Input Method": "CSV",
        "Callable": lambda: validate_batch_input(
            blank_categorical_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "Missing categorical value"
        ),
    },
    {
        "Test Name": "Missing direct-entry predictor",
        "Input Method": "Direct Entry",
        "Callable": lambda: create_input_dataframe(
            missing_direct_feature_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "Missing required predictors"
        ),
    },
    {
        "Test Name": "Unexpected direct-entry predictor",
        "Input Method": "Direct Entry",
        "Callable": lambda: create_input_dataframe(
            unexpected_direct_feature_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": (
            "Unexpected predictors received"
        ),
    },
    {
        "Test Name": "Invalid direct-entry numeric value",
        "Input Method": "Direct Entry",
        "Callable": lambda: create_input_dataframe(
            invalid_direct_numeric_case.copy()
        ),
        "Expected Outcome": "Rejected",
        "Expected Exception": ValueError,
        "Expected Message": None,
    },
]

# 7. Execute each validation test safely
def run_validation_test(
    test_case: dict,
) -> dict:
    """
    Run one validation test and compare the actual behavior
    with the expected acceptance or rejection.
    """

    test_callable: Callable = test_case["Callable"]
    expected_outcome = test_case["Expected Outcome"]
    expected_exception = test_case[
        "Expected Exception"
    ]
    expected_message = test_case[
        "Expected Message"
    ]

    actual_outcome = "Accepted"
    actual_exception_type = ""
    actual_message = ""

    try:
        test_callable()

    except Exception as error:
        actual_outcome = "Rejected"
        actual_exception_type = type(error).__name__
        actual_message = str(error)

    if expected_outcome == "Accepted":
        passed = actual_outcome == "Accepted"

    else:
        exception_match = (
            actual_outcome == "Rejected"
            and expected_exception is not None
            and actual_exception_type
            == expected_exception.__name__
        )

        message_match = (
            True
            if expected_message is None
            else expected_message.lower()
            in actual_message.lower()
        )

        passed = (
            exception_match
            and message_match
        )

    return {
        "Test Name": test_case["Test Name"],
        "Input Method": test_case["Input Method"],
        "Expected Outcome": expected_outcome,
        "Actual Outcome": actual_outcome,
        "Exception Type": actual_exception_type,
        "Validation Message": actual_message,
        "Result": "PASSED" if passed else "FAILED",
    }


input_validation_results = pd.DataFrame(
    [
        run_validation_test(test_case)
        for test_case in validation_test_cases
    ]
)

display(input_validation_results)

# 8. Summarize the validation results
passed_test_count = int(
    (
        input_validation_results["Result"]
        == "PASSED"
    ).sum()
)

failed_test_count = int(
    (
        input_validation_results["Result"]
        == "FAILED"
    ).sum()
)

input_validation_summary = pd.DataFrame(
    {
        "Validation Property": [
            "Total tests",
            "Passed tests",
            "Failed tests",
            "Accepted valid-input tests",
            "Rejected invalid-input tests",
        ],
        "Validated Value": [
            len(input_validation_results),
            passed_test_count,
            failed_test_count,
            int(
                (
                    input_validation_results[
                        "Expected Outcome"
                    ]
                    == "Accepted"
                ).sum()
            ),
            int(
                (
                    input_validation_results[
                        "Expected Outcome"
                    ]
                    == "Rejected"
                ).sum()
            ),
        ],
    }
)

display(input_validation_summary)


# 9. Stop immediately if any test failed

failed_input_tests = input_validation_results[
    input_validation_results["Result"]
    == "FAILED"
]

if not failed_input_tests.empty:
    display(failed_input_tests)

    raise AssertionError(
        f"{len(failed_input_tests)} input-validation "
        "test(s) did not behave as expected."
    )

# 10. Preserve results for final output files

step_4_input_validation_results = (
    input_validation_results.copy()
)

step_4_input_validation_summary = (
    input_validation_summary.copy()
)


print("\n" + "=" * 72)
print("STEP 4 RESULT: PASSED")
print(
    f"All {len(input_validation_results)} valid and invalid "
    "input tests behaved as expected."
)
print(
    "Invalid records were rejected before preprocessing "
    "or model prediction."
)
print("=" * 72)


## Step 5: Downloadable Output Validation

The Streamlit application allows users to download:

1. A single-patient screening result containing the prediction,
   threshold decisions, and all 43 entered predictors.
2. Batch screening results for every uploaded record.
3. Record-level prediction factors showing the five strongest
   increasing and five strongest reducing factors.

This step verifies that the downloadable CSV structures are complete,
readable, traceable to the original predictor names, and consistent
with the finalized prediction results. It also confirms that coded
values such as age bands and discharge dispositions are exported with
user-friendly labels.

In [ ]:
from io import StringIO

from ui.form_config import get_option_label

# 1. User-friendly classification mappings
STANDARD_REVIEW_RESULT_MAP = {
    "Flagged at Main Threshold": "Review Recommended",
    "Not Flagged at Main Threshold": (
        "Standard Review Not Triggered"
    ),
}

ADDITIONAL_SCREENING_RESULT_MAP = {
    "Flagged for Screening": (
        "Additional Screening Recommended"
    ),
    "Not Flagged": "No Additional Screening Flag",
}

# 2. Expected downloadable column structures
EXPECTED_BATCH_SCREENING_COLUMNS = [
    "Record",
    "Estimated 30-Day Readmission Risk (%)",
    "Standard Review Cutoff",
    "Standard Review Result",
    "Additional Screening Cutoff",
    "Additional Screening Result",
]

EXPECTED_SINGLE_RESULT_COLUMNS = [
    "Estimated 30-Day Readmission Risk (%)",
    "Standard Review Cutoff",
    "Standard Review Result",
    "Additional Screening Cutoff",
    "Additional Screening Result",
    *expected_feature_order,
]

EXPECTED_FACTOR_COLUMNS = [
    "Record Number",
    "Readmission Probability (%)",
    "Direction",
    "Factor Rank",
    "Feature",
    "Patient Value",
    "Original Feature",
]

# 3. Create the batch screening download table
batch_screening_download = (
    batch_prediction_results[
        [
            "Record Number",
            "Readmission Probability (%)",
            "Main Threshold",
            "Main Classification",
            "Recall-Focused Threshold",
            "Recall-Focused Classification",
        ]
    ]
    .rename(
        columns={
            "Record Number": "Record",
            "Readmission Probability (%)": (
                "Estimated 30-Day Readmission Risk (%)"
            ),
            "Main Threshold": (
                "Standard Review Cutoff"
            ),
            "Main Classification": (
                "Standard Review Result"
            ),
            "Recall-Focused Threshold": (
                "Additional Screening Cutoff"
            ),
            "Recall-Focused Classification": (
                "Additional Screening Result"
            ),
        }
    )
    .copy()
)

batch_screening_download[
    "Standard Review Result"
] = (
    batch_screening_download[
        "Standard Review Result"
    ]
    .map(STANDARD_REVIEW_RESULT_MAP)
)

batch_screening_download[
    "Additional Screening Result"
] = (
    batch_screening_download[
        "Additional Screening Result"
    ]
    .map(ADDITIONAL_SCREENING_RESULT_MAP)
)

batch_screening_download = batch_screening_download[
    EXPECTED_BATCH_SCREENING_COLUMNS
]

display(batch_screening_download)

# 4. Create the single-patient screening download table
single_patient_values = {
    feature: sample_input_validated.iloc[0][feature]
    for feature in expected_feature_order
}

single_patient_prediction = predict_readmission(
    single_patient_values
)

single_standard_result = (
    "Review Recommended"
    if single_patient_prediction[
        "main_threshold_prediction"
    ] == 1
    else "Standard Review Not Triggered"
)

single_additional_result = (
    "Additional Screening Recommended"
    if single_patient_prediction[
        "recall_focused_prediction"
    ] == 1
    else "No Additional Screening Flag"
)

single_screening_row = {
    "Estimated 30-Day Readmission Risk (%)": round(
        float(
            single_patient_prediction[
                "probability_percentage"
            ]
        ),
        8,
    ),
    "Standard Review Cutoff": float(
        single_patient_prediction[
            "main_threshold"
        ]
    ),
    "Standard Review Result": (
        single_standard_result
    ),
    "Additional Screening Cutoff": float(
        single_patient_prediction[
            "recall_focused_threshold"
        ]
    ),
    "Additional Screening Result": (
        single_additional_result
    ),
    **single_patient_values,
}

single_screening_download = pd.DataFrame(
    [single_screening_row],
    columns=EXPECTED_SINGLE_RESULT_COLUMNS,
)

display(single_screening_download)

# 5. Create the readable explanation-factor download table
def readable_download_value(
    original_feature: str,
    patient_value,
) -> str:
    """Match the user-friendly value formatting used by app.py."""

    if original_feature in input_schema["categorical_features"]:
        return get_option_label(
            original_feature,
            patient_value,
        )

    if original_feature == "time_in_hospital":
        return f"{patient_value} day(s)"

    return str(patient_value)


factor_download = batch_explanation_results[
    EXPECTED_FACTOR_COLUMNS
].copy()

factor_download[
    "Readmission Probability (%)"
] = (
    pd.to_numeric(
        factor_download[
            "Readmission Probability (%)"
        ],
        errors="raise",
    )
    .round(4)
)

factor_download["Patient Value"] = (
    factor_download.apply(
        lambda row: readable_download_value(
            original_feature=str(
                row["Original Feature"]
            ),
            patient_value=row["Patient Value"],
        ),
        axis=1,
    )
)

factor_download = factor_download.sort_values(
    [
        "Record Number",
        "Direction",
        "Factor Rank",
    ]
).reset_index(drop=True)

display(factor_download)

# 6. Convert each table to CSV download payloads
def create_csv_payload(
    dataframe: pd.DataFrame,
) -> tuple[bytes, pd.DataFrame]:
    """
    Convert a DataFrame to a UTF-8 CSV download payload and
    reload it to verify that the exported CSV remains readable.
    """

    csv_text = dataframe.to_csv(
        index=False,
        encoding="utf-8",
    )

    csv_bytes = csv_text.encode("utf-8")

    reloaded_dataframe = pd.read_csv(
        StringIO(csv_text)
    )

    return csv_bytes, reloaded_dataframe


(
    single_screening_csv_bytes,
    single_screening_reloaded,
) = create_csv_payload(
    single_screening_download
)

(
    batch_screening_csv_bytes,
    batch_screening_reloaded,
) = create_csv_payload(
    batch_screening_download
)

(
    factor_csv_bytes,
    factor_download_reloaded,
) = create_csv_payload(
    factor_download
)

# ------------------------------------------------------------
# 7. Validate probabilities and threshold values
# ------------------------------------------------------------
single_download_probability = float(
    single_screening_download[
        "Estimated 30-Day Readmission Risk (%)"
    ].iloc[0]
)

expected_single_probability = float(
    single_patient_prediction[
        "probability_percentage"
    ]
)

single_probability_difference = abs(
    single_download_probability
    - expected_single_probability
)

single_probability_match = np.isclose(
    single_download_probability,
    expected_single_probability,
    atol=1e-7,
    rtol=0,
)


# The batch download is created from batch_prediction_results.
# Compare every downloaded probability with that exact source.
batch_download_probabilities = (
    batch_screening_download[
        "Estimated 30-Day Readmission Risk (%)"
    ]
    .astype(float)
    .to_numpy()
)

expected_batch_probabilities = (
    batch_prediction_results[
        "Readmission Probability (%)"
    ]
    .astype(float)
    .to_numpy()
)

batch_probability_differences = np.abs(
    batch_download_probabilities
    - expected_batch_probabilities
)

batch_probability_match = np.allclose(
    batch_download_probabilities,
    expected_batch_probabilities,
    atol=1e-12,
    rtol=0,
)


probability_validation_table = pd.DataFrame(
    {
        "Download Type": [
            "Single-patient result",
            *[
                f"Batch record {record_number}"
                for record_number in range(
                    1,
                    len(batch_download_probabilities) + 1,
                )
            ],
        ],
        "Downloaded Probability (%)": [
            single_download_probability,
            *batch_download_probabilities.tolist(),
        ],
        "Expected Probability (%)": [
            expected_single_probability,
            *expected_batch_probabilities.tolist(),
        ],
        "Absolute Difference": [
            single_probability_difference,
            *batch_probability_differences.tolist(),
        ],
        "Result": [
            "PASSED" if single_probability_match else "FAILED",
            *[
                "PASSED"
                if difference <= 1e-12
                else "FAILED"
                for difference in batch_probability_differences
            ],
        ],
    }
)

display(
    probability_validation_table.style.format(
        {
            "Downloaded Probability (%)": "{:.8f}",
            "Expected Probability (%)": "{:.8f}",
            "Absolute Difference": "{:.12f}",
        }
    )
)


# ------------------------------------------------------------
# 8. Validate explanation factor counts and rankings
# ------------------------------------------------------------
expected_factor_directions = {
    "Increases estimated readmission risk",
    "Reduces estimated readmission risk",
}

factor_count_table = (
    factor_download
    .groupby(
        [
            "Record Number",
            "Direction",
        ]
    )
    .size()
    .reset_index(name="Factor Count")
)

display(factor_count_table)


increasing_factor_counts = factor_count_table[
    factor_count_table["Direction"]
    == "Increases estimated readmission risk"
]

reducing_factor_counts = factor_count_table[
    factor_count_table["Direction"]
    == "Reduces estimated readmission risk"
]

increasing_factor_count_match = (
    len(increasing_factor_counts)
    == len(batch_prediction_results)
    and (
        increasing_factor_counts["Factor Count"]
        == 5
    ).all()
)

reducing_factor_count_match = (
    len(reducing_factor_counts)
    == len(batch_prediction_results)
    and (
        reducing_factor_counts["Factor Count"]
        == 5
    ).all()
)

factor_directions_match = (
    set(factor_download["Direction"].unique())
    == expected_factor_directions
)

factor_rank_match = True

for (
    record_number,
    direction,
), factor_group in factor_download.groupby(
    [
        "Record Number",
        "Direction",
    ]
):
    observed_ranks = sorted(
        factor_group["Factor Rank"]
        .astype(int)
        .tolist()
    )

    if observed_ranks != [1, 2, 3, 4, 5]:
        factor_rank_match = False
        break

# Validate user-friendly factor values and traceability
factor_values_are_readable = (
    factor_download["Patient Value"]
    .map(lambda value: isinstance(value, str) and bool(value.strip()))
    .all()
)

original_features_retained = (
    factor_download["Original Feature"].notna().all()
    and factor_download["Original Feature"].astype(str).str.strip().ne("").all()
)

factor_probabilities_rounded = (
    factor_download["Readmission Probability (%)"]
    .map(lambda value: float(value) == round(float(value), 4))
    .all()
)

age_factor_values = factor_download.loc[
    factor_download["Original Feature"] == "age",
    "Patient Value",
].astype(str).tolist()

age_values_are_readable = (
    len(age_factor_values) > 0
    and all(
        "years" in value.lower()
        for value in age_factor_values
    )
)

discharge_factor_values = factor_download.loc[
    factor_download["Original Feature"]
    == "discharge_disposition_id",
    "Patient Value",
].astype(str).tolist()

discharge_values_are_readable = (
    len(discharge_factor_values) > 0
    and all(
        not value.strip().replace(".", "", 1).isdigit()
        for value in discharge_factor_values
    )
)

# 9. Validate exported CSV round trips
single_round_trip_columns_match = (
    single_screening_reloaded.columns.tolist()
    == EXPECTED_SINGLE_RESULT_COLUMNS
)

batch_round_trip_columns_match = (
    batch_screening_reloaded.columns.tolist()
    == EXPECTED_BATCH_SCREENING_COLUMNS
)

factor_round_trip_columns_match = (
    factor_download_reloaded.columns.tolist()
    == EXPECTED_FACTOR_COLUMNS
)

single_round_trip_shape_match = (
    single_screening_reloaded.shape
    == single_screening_download.shape
)

batch_round_trip_shape_match = (
    batch_screening_reloaded.shape
    == batch_screening_download.shape
)

factor_round_trip_shape_match = (
    factor_download_reloaded.shape
    == factor_download.shape
)

# 10. Run all downloadable-output validation checks
step_5_validation_checks = {
    "Batch screening download contains expected columns": (
        batch_screening_download.columns.tolist()
        == EXPECTED_BATCH_SCREENING_COLUMNS
    ),
    "Single screening download contains expected columns": (
        single_screening_download.columns.tolist()
        == EXPECTED_SINGLE_RESULT_COLUMNS
    ),
    "Single screening result contains 43 predictor values": (
        len(
            [
                column
                for column in expected_feature_order
                if column
                in single_screening_download.columns
            ]
        )
        == 43
    ),
    "Factor download contains expected columns": (
        factor_download.columns.tolist()
        == EXPECTED_FACTOR_COLUMNS
    ),
    "Factor download patient values are readable": (
        factor_values_are_readable
    ),
    "Factor download retains original feature names": (
        original_features_retained
    ),
    "Factor download probabilities are rounded to four decimals": (
        factor_probabilities_rounded
    ),
    "Age factors use readable year-range labels": (
        age_values_are_readable
    ),
    "Discharge-disposition factors use readable labels": (
        discharge_values_are_readable
    ),
    "Single download probability matches model output": (
        single_probability_match
    ),
    "Batch download probability matches model output": (
        batch_probability_match
    ),
    "Single download uses the 0.50 standard cutoff": (
        float(
            single_screening_download[
                "Standard Review Cutoff"
            ].iloc[0]
        )
        == 0.50
    ),
    "Single download uses the 0.45 screening cutoff": (
        float(
            single_screening_download[
                "Additional Screening Cutoff"
            ].iloc[0]
        )
        == 0.45
    ),
    "Batch download contains valid standard-review wording": (
        batch_screening_download[
            "Standard Review Result"
        ]
        .isin(
            [
                "Review Recommended",
                "Standard Review Not Triggered",
            ]
        )
        .all()
    ),
    "Batch download contains valid screening wording": (
        batch_screening_download[
            "Additional Screening Result"
        ]
        .isin(
            [
                "Additional Screening Recommended",
                "No Additional Screening Flag",
            ]
        )
        .all()
    ),
    "Each record contains five increasing factors": (
    increasing_factor_count_match
),
"Each record contains five reducing factors": (
    reducing_factor_count_match
),
    "Factor directions are complete": (
        factor_directions_match
    ),
    "Factor rankings contain values one through five": (
        factor_rank_match
    ),
    "Single screening CSV reload preserves columns": (
        single_round_trip_columns_match
    ),
    "Batch screening CSV reload preserves columns": (
        batch_round_trip_columns_match
    ),
    "Factor CSV reload preserves columns": (
        factor_round_trip_columns_match
    ),
    "Single screening CSV reload preserves shape": (
        single_round_trip_shape_match
    ),
    "Batch screening CSV reload preserves shape": (
        batch_round_trip_shape_match
    ),
    "Factor CSV reload preserves shape": (
        factor_round_trip_shape_match
    ),
    "Single screening CSV payload is not empty": (
        len(single_screening_csv_bytes) > 0
    ),
    "Batch screening CSV payload is not empty": (
        len(batch_screening_csv_bytes) > 0
    ),
    "Factor CSV payload is not empty": (
        len(factor_csv_bytes) > 0
    ),
}

step_5_validation_table = pd.DataFrame(
    [
        {
            "Validation Check": check_name,
            "Result": "PASSED" if passed else "FAILED",
        }
        for check_name, passed
        in step_5_validation_checks.items()
    ]
)

display(step_5_validation_table)

# 11. Display downloadable-file summary

download_file_summary = pd.DataFrame(
    {
        "Download File": [
            "Single-patient screening result",
            "Batch screening results",
            "Prediction-factor results",
        ],
        "Rows": [
            len(single_screening_download),
            len(batch_screening_download),
            len(factor_download),
        ],
        "Columns": [
            len(single_screening_download.columns),
            len(batch_screening_download.columns),
            len(factor_download.columns),
        ],
        "CSV Size (bytes)": [
            len(single_screening_csv_bytes),
            len(batch_screening_csv_bytes),
            len(factor_csv_bytes),
        ],
    }
)

display(download_file_summary)

# 12. Stop immediately if any output check failed
failed_output_checks = [
    check_name
    for check_name, passed
    in step_5_validation_checks.items()
    if not passed
]

if failed_output_checks:
    raise AssertionError(
        "Download-output validation failed:\n"
        + "\n".join(
            f"- {check}"
            for check in failed_output_checks
        )
    )

# 13. Preserve results for later notebook outputs
step_5_single_screening_download = (
    single_screening_download.copy()
)

step_5_batch_screening_download = (
    batch_screening_download.copy()
)

step_5_factor_download = factor_download.copy()

step_5_validation_summary = (
    step_5_validation_table.copy()
)

step_5_download_file_summary = (
    download_file_summary.copy()
)


print("\n" + "=" * 72)
print("STEP 5 RESULT: PASSED")
print(
    "All downloadable screening and explanation files "
    "contain the required columns, values, and records."
)
print(
    "All CSV files were exported and reloaded successfully."
)
print("=" * 72)

## Step 6: Streamlit Application Structure and Asset Validation

This step validates the complete application package used for local and
cloud deployment.

The checks cover:

- required Python source files
- Python syntax for all application modules
- all eight application pages, including Application Validation
- all three prediction input methods
- guided-form, confirmation, prediction, explanation, and validation components
- the professional Streamlit theme
- blank and synthetic CSV templates
- all 22 approved saved figures
- public-facing wording
- final threshold references
- readable single-patient and batch factor-download logic
- saved Notebook 09 evidence-path integration

In [ ]:

import ast
import tomllib
from pathlib import Path

import pandas as pd

# 1. Define required application paths
APP_PATH = PROJECT_ROOT / "app.py"
PREDICTION_SERVICE_PATH = (
    PROJECT_ROOT / "prediction_service.py"
)
CUSTOM_TRANSFORMERS_PATH = (
    PROJECT_ROOT / "custom_transformers.py"
)

UI_INIT_PATH = PROJECT_ROOT / "ui" / "__init__.py"
UI_FORM_CONFIG_PATH = (
    PROJECT_ROOT / "ui" / "form_config.py"
)
UI_COMPONENTS_PATH = (
    PROJECT_ROOT / "ui" / "components.py"
)
UI_STYLES_PATH = (
    PROJECT_ROOT / "ui" / "styles.py"
)

STREAMLIT_CONFIG_PATH = (
    PROJECT_ROOT / ".streamlit" / "config.toml"
)

BLANK_TEMPLATE_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "patient_input_template.csv"
)

SYNTHETIC_SAMPLE_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "sample_patient_input.csv"
)

FIGURES_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

README_PATH = PROJECT_ROOT / "README.md"
USER_GUIDE_PATH = (
    PROJECT_ROOT / "STREAMLIT_APP_USER_GUIDE.md"
)

VALIDATION_JSON_SUMMARY_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_9_streamlit_validation_summary.json"
)

VALIDATION_OVERALL_SUMMARY_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "notebook_9_streamlit_validation_overall_summary.csv"
)

VALIDATION_STEP_SUMMARY_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "notebook_9_streamlit_validation_step_summary.csv"
)

VALIDATION_CHECKS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "notebook_9_streamlit_validation_checks.csv"
)

VALIDATION_PARITY_RESULTS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "notebook_9_direct_csv_parity_results.csv"
)

VALIDATION_INVALID_INPUT_RESULTS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "notebook_9_invalid_input_test_results.csv"
)

VALIDATION_DOWNLOAD_RESULTS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "notebook_9_download_validation_results.csv"
)

VALIDATION_FIGURE_RESULTS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "notebook_9_approved_figure_validation.csv"
)


required_application_files = {
    "Streamlit application": APP_PATH,
    "Prediction service": PREDICTION_SERVICE_PATH,
    "Custom transformers": CUSTOM_TRANSFORMERS_PATH,
    "UI package initializer": UI_INIT_PATH,
    "Guided-form configuration": UI_FORM_CONFIG_PATH,
    "Reusable UI components": UI_COMPONENTS_PATH,
    "Application styles": UI_STYLES_PATH,
    "Streamlit theme configuration": STREAMLIT_CONFIG_PATH,
    "Blank patient-input template": BLANK_TEMPLATE_PATH,
    "Synthetic patient sample": SYNTHETIC_SAMPLE_PATH,
    "README": README_PATH,
    "Streamlit application user guide": USER_GUIDE_PATH,
    "Validation JSON summary": VALIDATION_JSON_SUMMARY_PATH,
    "Validation overall summary": VALIDATION_OVERALL_SUMMARY_PATH,
    "Validation step summary": VALIDATION_STEP_SUMMARY_PATH,
    "Detailed validation checks": VALIDATION_CHECKS_PATH,
    "Prediction parity validation": VALIDATION_PARITY_RESULTS_PATH,
    "Invalid-input validation": VALIDATION_INVALID_INPUT_RESULTS_PATH,
    "Download validation": VALIDATION_DOWNLOAD_RESULTS_PATH,
    "Approved-figure validation": VALIDATION_FIGURE_RESULTS_PATH,
}

# 2. Confirm all required files exist
required_file_rows = []

for file_description, file_path in required_application_files.items():
    required_file_rows.append(
        {
            "Required File": file_description,
            "Relative Path": str(
                file_path.relative_to(PROJECT_ROOT)
            ),
            "Exists": file_path.exists(),
            "Result": (
                "PASSED"
                if file_path.exists()
                else "FAILED"
            ),
        }
    )

required_file_table = pd.DataFrame(
    required_file_rows
)

display(required_file_table)


missing_application_files = required_file_table[
    required_file_table["Exists"] == False
]

if not missing_application_files.empty:
    raise FileNotFoundError(
        "Required application files are missing:\n"
        + "\n".join(
            missing_application_files[
                "Relative Path"
            ].tolist()
        )
    )

# 3. Validate Python syntax without importing Streamlit app.py
python_source_paths = {
    "app.py": APP_PATH,
    "prediction_service.py": PREDICTION_SERVICE_PATH,
    "custom_transformers.py": CUSTOM_TRANSFORMERS_PATH,
    "ui/__init__.py": UI_INIT_PATH,
    "ui/form_config.py": UI_FORM_CONFIG_PATH,
    "ui/components.py": UI_COMPONENTS_PATH,
    "ui/styles.py": UI_STYLES_PATH,
}

syntax_validation_rows = []

for source_name, source_path in python_source_paths.items():
    source_text = source_path.read_text(
        encoding="utf-8"
    )

    syntax_error_message = ""

    try:
        compile(
            source_text,
            str(source_path),
            "exec",
        )
        syntax_passed = True

    except SyntaxError as error:
        syntax_passed = False
        syntax_error_message = str(error)

    syntax_validation_rows.append(
        {
            "Python Source": source_name,
            "Syntax Valid": syntax_passed,
            "Error Message": syntax_error_message,
            "Result": (
                "PASSED"
                if syntax_passed
                else "FAILED"
            ),
        }
    )

syntax_validation_table = pd.DataFrame(
    syntax_validation_rows
)

display(syntax_validation_table)

# 4. Parse application source safely
app_source = APP_PATH.read_text(
    encoding="utf-8"
)

app_syntax_tree = ast.parse(
    app_source,
    filename=str(APP_PATH),
)

app_function_names = {
    node.name
    for node in ast.walk(app_syntax_tree)
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    )
}

# 5. Validate all eight application pages
expected_page_renderers = {
    "Overview": "render_project_overview",
    "Data Explorer": "render_dataset_summary",
    "Model Development": "render_model_development",
    "Model Performance": "render_final_evaluation",
    "Risk Insights": "render_explainability",
    "Application Validation": "render_application_validation",
    "Saved Figures": "render_saved_figures",
    "New Prediction": "render_prediction",
}

page_validation_rows = []

for page_label, renderer_name in expected_page_renderers.items():
    label_present = page_label in app_source
    renderer_present = (
        renderer_name in app_function_names
    )
    renderer_called = (
        f"{renderer_name}()" in app_source
    )

    page_validation_rows.append(
        {
            "Navigation Page": page_label,
            "Renderer Function": renderer_name,
            "Navigation Label Present": label_present,
            "Renderer Defined": renderer_present,
            "Renderer Called": renderer_called,
            "Result": (
                "PASSED"
                if (
                    label_present
                    and renderer_present
                    and renderer_called
                )
                else "FAILED"
            ),
        }
    )

page_validation_table = pd.DataFrame(
    page_validation_rows
)

display(page_validation_table)

# 6. Validate all three prediction input methods
expected_input_methods = [
    "Enter One Patient",
    "Upload Multiple Records",
    "Use Sample Record",
]

input_method_rows = []

for input_method in expected_input_methods:
    method_present = input_method in app_source

    input_method_rows.append(
        {
            "Input Method": input_method,
            "Found in Application": method_present,
            "Result": (
                "PASSED"
                if method_present
                else "FAILED"
            ),
        }
    )

input_method_table = pd.DataFrame(
    input_method_rows
)

display(input_method_table)

# 7. Validate essential page and prediction functions
required_app_functions = [
    "load_guided_form_schema",
    "initialize_single_patient_state",
    "invalidate_single_patient_results",
    "reset_single_patient_form",
    "render_single_patient_progress",
    "render_single_input_field",
    "readable_form_value",
    "create_readable_explanation_download",
    "render_single_patient_review",
    "calculate_single_patient_results",
    "render_single_patient_results",
    "render_single_patient_form",
    "render_batch_prediction",
    "load_sample_record_into_form",
    "load_validation_overall_summary",
    "load_validation_step_summary",
    "load_validation_checks",
    "load_validation_parity_results",
    "load_validation_invalid_input_results",
    "load_validation_download_results",
    "load_validation_figure_results",
    "render_application_validation",
]

app_function_validation_rows = []

for function_name in required_app_functions:
    function_present = (
        function_name in app_function_names
    )

    app_function_validation_rows.append(
        {
            "Required Application Function": function_name,
            "Function Defined": function_present,
            "Result": (
                "PASSED"
                if function_present
                else "FAILED"
            ),
        }
    )

app_function_validation_table = pd.DataFrame(
    app_function_validation_rows
)

display(app_function_validation_table)

# 8. Validate reusable UI components
components_source = UI_COMPONENTS_PATH.read_text(
    encoding="utf-8"
)

components_tree = ast.parse(
    components_source,
    filename=str(UI_COMPONENTS_PATH),
)

component_function_names = {
    node.name
    for node in ast.walk(components_tree)
    if isinstance(node, ast.FunctionDef)
}

required_component_functions = [
    "render_page_hero",
    "render_metric_card",
    "render_info_card",
    "render_three_step_workflow",
    "render_threshold_card",
    "render_key_message",
    "render_screening_status_card",
    "render_probability_scale",
    "render_factor_panel",
]

component_validation_rows = []

for function_name in required_component_functions:
    function_present = (
        function_name in component_function_names
    )

    component_validation_rows.append(
        {
            "Required UI Component": function_name,
            "Function Defined": function_present,
            "Result": (
                "PASSED"
                if function_present
                else "FAILED"
            ),
        }
    )

component_validation_table = pd.DataFrame(
    component_validation_rows
)

display(component_validation_table)

# 9. Validate the Streamlit theme configuration
with open(
    STREAMLIT_CONFIG_PATH,
    "rb",
) as config_file:
    streamlit_config = tomllib.load(config_file)

main_theme = streamlit_config.get(
    "theme",
    {},
)

sidebar_theme = main_theme.get(
    "sidebar",
    {},
)

theme_validation_checks = {
    "Main theme uses a light base": (
        main_theme.get("base") == "light"
    ),
    "Main primary color is configured": (
        bool(main_theme.get("primaryColor"))
    ),
    "Main background color is configured": (
        bool(main_theme.get("backgroundColor"))
    ),
    "Main text color is configured": (
        bool(main_theme.get("textColor"))
    ),
    "Sidebar background color is configured": (
        bool(sidebar_theme.get("backgroundColor"))
    ),
    "Sidebar text color is configured": (
        bool(sidebar_theme.get("textColor"))
    ),
    "Sidebar border is enabled": (
        main_theme.get("showSidebarBorder")
        is True
    ),
}

theme_validation_table = pd.DataFrame(
    [
        {
            "Theme Check": check_name,
            "Result": (
                "PASSED"
                if passed
                else "FAILED"
            ),
        }
        for check_name, passed
        in theme_validation_checks.items()
    ]
)

display(theme_validation_table)


theme_summary = pd.DataFrame(
    {
        "Theme Property": [
            "Base",
            "Primary color",
            "Background color",
            "Text color",
            "Sidebar background",
            "Sidebar text color",
        ],
        "Configured Value": [
            main_theme.get("base"),
            main_theme.get("primaryColor"),
            main_theme.get("backgroundColor"),
            main_theme.get("textColor"),
            sidebar_theme.get("backgroundColor"),
            sidebar_theme.get("textColor"),
        ],
    }
)

display(theme_summary)

# 10. Validate blank and synthetic CSV files
blank_template = pd.read_csv(
    BLANK_TEMPLATE_PATH
)

synthetic_template = pd.read_csv(
    SYNTHETIC_SAMPLE_PATH
)

template_validation_checks = {
    "Blank template contains 43 columns": (
        blank_template.shape[1] == 43
    ),
    "Blank template column order matches schema": (
        blank_template.columns.tolist()
        == expected_feature_order
    ),
    "Blank template contains no completed records": (
        len(blank_template) == 0
    ),
    "Synthetic sample contains 43 columns": (
        synthetic_template.shape[1] == 43
    ),
    "Synthetic sample column order matches schema": (
        synthetic_template.columns.tolist()
        == expected_feature_order
    ),
    "Synthetic sample contains at least one record": (
        len(synthetic_template) >= 1
    ),
}

template_validation_table = pd.DataFrame(
    [
        {
            "Template Check": check_name,
            "Result": (
                "PASSED"
                if passed
                else "FAILED"
            ),
        }
        for check_name, passed
        in template_validation_checks.items()
    ]
)

display(template_validation_table)

# 11. Extract APPROVED_FIGURES from app.py safely
def extract_literal_assignment(
    syntax_tree: ast.AST,
    variable_name: str,
):
    """
    Locate a top-level literal assignment and evaluate it
    without importing or executing the source file.
    """

    for node in syntax_tree.body:
        if isinstance(node, ast.Assign):
            target_names = [
                target.id
                for target in node.targets
                if isinstance(target, ast.Name)
            ]

            if variable_name in target_names:
                return ast.literal_eval(node.value)

        if isinstance(node, ast.AnnAssign):
            if (
                isinstance(node.target, ast.Name)
                and node.target.id == variable_name
            ):
                return ast.literal_eval(node.value)

    raise KeyError(
        f"{variable_name} was not found in app.py."
    )


approved_figures = extract_literal_assignment(
    app_syntax_tree,
    "APPROVED_FIGURES",
)

approved_figure_rows = []

for figure_label, figure_information in approved_figures.items():
    figure_filename = figure_information[
        "filename"
    ]

    figure_path = (
        FIGURES_DIRECTORY
        / figure_filename
    )

    approved_figure_rows.append(
        {
            "Figure Label": figure_label,
            "Category": figure_information[
                "category"
            ],
            "Filename": figure_filename,
            "File Exists": figure_path.exists(),
            "Result": (
                "PASSED"
                if figure_path.exists()
                else "FAILED"
            ),
        }
    )

approved_figure_table = pd.DataFrame(
    approved_figure_rows
)

display(approved_figure_table)


approved_figure_filenames = approved_figure_table[
    "Filename"
].tolist()

figure_validation_checks = {
    "Application defines 22 approved figures": (
        len(approved_figures) == 22
    ),
    "All approved figure labels are unique": (
        len(set(approved_figures.keys()))
        == len(approved_figures)
    ),
    "All approved figure filenames are unique": (
        len(set(approved_figure_filenames))
        == len(approved_figure_filenames)
    ),
    "All approved figure files exist": (
        approved_figure_table[
            "File Exists"
        ].all()
    ),
}

figure_validation_table = pd.DataFrame(
    [
        {
            "Figure Validation Check": check_name,
            "Result": (
                "PASSED"
                if passed
                else "FAILED"
            ),
        }
        for check_name, passed
        in figure_validation_checks.items()
    ]
)

display(figure_validation_table)

# 12. Validate final threshold and public-facing wording
public_source_paths = {
    "app.py": APP_PATH,
    "README.md": README_PATH,
    "STREAMLIT_APP_USER_GUIDE.md": (
        USER_GUIDE_PATH
    ),
    "ui/form_config.py": UI_FORM_CONFIG_PATH,
    "ui/components.py": UI_COMPONENTS_PATH,
    "ui/styles.py": UI_STYLES_PATH,
}

public_wording_rows = []

for source_name, source_path in public_source_paths.items():
    source_content = source_path.read_text(
        encoding="utf-8"
    )

    professor_reference_found = (
        "professor" in source_content.lower()
    )

    public_wording_rows.append(
        {
            "Public File": source_name,
            "Professor Reference Found": (
                professor_reference_found
            ),
            "Result": (
                "FAILED"
                if professor_reference_found
                else "PASSED"
            ),
        }
    )

public_wording_table = pd.DataFrame(
    public_wording_rows
)

display(public_wording_table)


wording_and_threshold_checks = {
    "Application references the 0.50 cutoff": (
        "0.50" in app_source
    ),
    "Application references the 0.45 cutoff": (
        "0.45" in app_source
    ),
    "Application includes Standard Review Not Triggered": (
        "Standard Review Not Triggered"
        in app_source
    ),
    "Application includes Review Recommended": (
        "Review Recommended"
        in app_source
    ),
    "Application includes Additional Screening Recommended": (
        "Additional Screening Recommended"
        in app_source
    ),
    "Application includes de-identification guidance": (
        "de-identified" in app_source.lower()
    ),
    "Application includes academic prototype disclaimer": (
        "academic decision-support prototype"
        in app_source.lower()
    ),
    "No professor attribution appears in public files": (
        not public_wording_table[
            "Professor Reference Found"
        ].any()
    ),
}

wording_validation_table = pd.DataFrame(
    [
        {
            "Wording Check": check_name,
            "Result": (
                "PASSED"
                if passed
                else "FAILED"
            ),
        }
        for check_name, passed
        in wording_and_threshold_checks.items()
    ]
)

display(wording_validation_table)

# 13. Validate the confirmation, readable-download, and validation-page additions
confirmation_checkbox_configured = (
    "single_patient_values_confirmed" in app_source
    and "I have reviewed all 43 predictor values" in app_source
    and "st.checkbox(" in app_source
)

calculate_requires_confirmation = (
    "disabled=not values_confirmed" in app_source
)

input_change_clears_confirmation = (
    'st.session_state["single_patient_values_confirmed"] = False'
    in app_source
    and "on_change=invalidate_single_patient_results"
    in app_source
)

readable_factor_downloads_configured = (
    "create_readable_explanation_download"
    in app_function_names
    and app_source.count(
        "create_readable_explanation_download("
    ) >= 3
    and '"Original Feature"' in app_source
)

validation_evidence_constants = [
    "VALIDATION_JSON_SUMMARY_PATH",
    "VALIDATION_OVERALL_SUMMARY_PATH",
    "VALIDATION_STEP_SUMMARY_PATH",
    "VALIDATION_CHECKS_PATH",
    "VALIDATION_PARITY_RESULTS_PATH",
    "VALIDATION_INVALID_INPUT_RESULTS_PATH",
    "VALIDATION_DOWNLOAD_RESULTS_PATH",
    "VALIDATION_FIGURE_RESULTS_PATH",
]

validation_tab_labels = [
    "Prediction Consistency",
    "Input Validation",
    "Downloads",
    "Application Structure",
    "Approved Figures",
]

validation_dashboard_configured = (
    "render_application_validation"
    in app_function_names
    and all(
        constant_name in app_source
        for constant_name in validation_evidence_constants
    )
    and all(
        tab_label in app_source
        for tab_label in validation_tab_labels
    )
)

# 14. Combine all final Step 6 checks
step_6_validation_checks = {
    "All required application files exist": (
        required_file_table["Exists"].all()
    ),
    "All Python application files have valid syntax": (
        syntax_validation_table[
            "Syntax Valid"
        ].all()
    ),
    "All eight application pages are configured": (
        (
            page_validation_table["Result"]
            == "PASSED"
        ).all()
    ),
    "All three prediction input methods are configured": (
        (
            input_method_table["Result"]
            == "PASSED"
        ).all()
    ),
    "All essential app functions are defined": (
        (
            app_function_validation_table[
                "Result"
            ]
            == "PASSED"
        ).all()
    ),
    "All reusable UI components are defined": (
        (
            component_validation_table[
                "Result"
            ]
            == "PASSED"
        ).all()
    ),
    "Professional Streamlit theme is valid": (
        all(theme_validation_checks.values())
    ),
    "Blank and synthetic templates are valid": (
        all(template_validation_checks.values())
    ),
    "All 22 approved figures are available": (
        all(figure_validation_checks.values())
    ),
    "Public wording and threshold references are valid": (
        all(
            wording_and_threshold_checks.values()
        )
    ),
    "Direct-entry confirmation checkbox is configured": (
        confirmation_checkbox_configured
    ),
    "Calculate action requires confirmed values": (
        calculate_requires_confirmation
    ),
    "Changing inputs clears confirmation and prior results": (
        input_change_clears_confirmation
    ),
    "Readable factor downloads are configured": (
        readable_factor_downloads_configured
    ),
    "Application Validation dashboard uses saved evidence": (
        validation_dashboard_configured
    ),
}

step_6_validation_table = pd.DataFrame(
    [
        {
            "Validation Check": check_name,
            "Result": (
                "PASSED"
                if passed
                else "FAILED"
            ),
        }
        for check_name, passed
        in step_6_validation_checks.items()
    ]
)

display(step_6_validation_table)

# 15. Stop immediately if any Step 6 check failed
failed_step_6_checks = [
    check_name
    for check_name, passed
    in step_6_validation_checks.items()
    if not passed
]

if failed_step_6_checks:
    raise AssertionError(
        "Application-structure validation failed:\n"
        + "\n".join(
            f"- {check}"
            for check in failed_step_6_checks
        )
    )

# 16. Preserve results for the final notebook summary
step_6_required_file_table = (
    required_file_table.copy()
)

step_6_page_validation_table = (
    page_validation_table.copy()
)

step_6_figure_validation_table = (
    figure_validation_table.copy()
)

step_6_approved_figure_table = (
    approved_figure_table.copy()
)

step_6_validation_summary = (
    step_6_validation_table.copy()
)


print("\n" + "=" * 72)
print("STEP 6 RESULT: PASSED")
print(
    "The complete Streamlit application structure, theme, "
    "templates, pages, input methods, and UI components are valid."
)
print(
    f"All {len(approved_figures)} approved saved figures were found."
)
print("=" * 72)

## Step 7: Final Validation Summary and Saved Evidence

This step consolidates all application-validation results and saves
permanent evidence for the capstone repository.

The final report includes:

- deployment-asset validation
- guided-form configuration validation
- direct-entry versus CSV prediction parity
- invalid-input and error-handling tests
- downloadable-output validation
- complete eight-page Streamlit application and saved-asset validation
- readable-download and confirmation-workflow validation
- Application Validation dashboard integration

No model retraining or threshold modification is performed.

In [ ]:
from datetime import datetime
import json


# 1. Confirm all required Step 1–6 result tables are available

required_result_objects = {
    "Step 1 deployment validation": "validation_table",
    "Step 2 guided-form validation": (
        "step_2_form_validation_table"
    ),
    "Step 3 prediction-parity validation": (
        "step_3_validation_summary"
    ),
    "Step 4 invalid-input validation": (
        "step_4_input_validation_results"
    ),
    "Step 5 download validation": (
        "step_5_validation_summary"
    ),
    "Step 6 application validation": (
        "step_6_validation_summary"
    ),
}

missing_result_objects = [
    object_name
    for object_name in required_result_objects.values()
    if object_name not in globals()
]

if missing_result_objects:
    raise NameError(
        "The following required validation results are missing:\n"
        + "\n".join(
            f"- {object_name}"
            for object_name in missing_result_objects
        )
        + "\nRun Steps 1–6 before executing Step 7."
    )

print("All Step 1–6 validation results are available.")


# 2. Normalize every step into one common table
def normalize_validation_table(
    dataframe: pd.DataFrame,
    *,
    step_number: int,
    step_name: str,
    check_column: str,
) -> pd.DataFrame:
    """
    Convert one validation table into the common final-report
    structure.
    """

    required_columns = {
        check_column,
        "Result",
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise KeyError(
            f"{step_name} is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    normalized = dataframe[
        [
            check_column,
            "Result",
        ]
    ].copy()

    normalized = normalized.rename(
        columns={
            check_column: "Validation Check",
        }
    )

    normalized.insert(
        0,
        "Step Number",
        step_number,
    )

    normalized.insert(
        1,
        "Validation Area",
        step_name,
    )

    return normalized


step_1_checks = normalize_validation_table(
    validation_table,
    step_number=1,
    step_name="Deployment Assets",
    check_column="Validation Check",
)

step_2_checks = normalize_validation_table(
    step_2_form_validation_table,
    step_number=2,
    step_name="Guided Form Configuration",
    check_column="Validation Check",
)

step_3_checks = normalize_validation_table(
    step_3_validation_summary,
    step_number=3,
    step_name="Prediction Parity",
    check_column="Validation Check",
)

step_4_checks = normalize_validation_table(
    step_4_input_validation_results,
    step_number=4,
    step_name="Invalid-Input Handling",
    check_column="Test Name",
)

step_5_checks = normalize_validation_table(
    step_5_validation_summary,
    step_number=5,
    step_name="Downloadable Outputs",
    check_column="Validation Check",
)

step_6_checks = normalize_validation_table(
    step_6_validation_summary,
    step_number=6,
    step_name="Application Structure",
    check_column="Validation Check",
)

# 3. Create the complete validation-check table
final_validation_checks = pd.concat(
    [
        step_1_checks,
        step_2_checks,
        step_3_checks,
        step_4_checks,
        step_5_checks,
        step_6_checks,
    ],
    ignore_index=True,
)

final_validation_checks.insert(
    3,
    "Passed",
    final_validation_checks["Result"].eq(
        "PASSED"
    ),
)

display(final_validation_checks)

# 4. Create a step-level summary
final_step_summary = (
    final_validation_checks
    .groupby(
        [
            "Step Number",
            "Validation Area",
        ],
        as_index=False,
    )
    .agg(
        Total_Checks=(
            "Validation Check",
            "size",
        ),
        Passed_Checks=(
            "Passed",
            "sum",
        ),
    )
)

final_step_summary[
    "Failed_Checks"
] = (
    final_step_summary["Total_Checks"]
    - final_step_summary["Passed_Checks"]
)

final_step_summary[
    "Pass_Rate_Percentage"
] = (
    final_step_summary["Passed_Checks"]
    / final_step_summary["Total_Checks"]
    * 100
)

final_step_summary[
    "Result"
] = np.where(
    final_step_summary["Failed_Checks"] == 0,
    "PASSED",
    "FAILED",
)

final_step_summary = final_step_summary.rename(
    columns={
        "Total_Checks": "Total Checks",
        "Passed_Checks": "Passed Checks",
        "Failed_Checks": "Failed Checks",
        "Pass_Rate_Percentage": (
            "Pass Rate (%)"
        ),
    }
)

display(
    final_step_summary.style.format(
        {
            "Pass Rate (%)": "{:.2f}",
        }
    )
)

# 5. Calculate overall validation totals
total_validation_checks = int(
    len(final_validation_checks)
)

total_passed_checks = int(
    final_validation_checks["Passed"].sum()
)

total_failed_checks = int(
    total_validation_checks
    - total_passed_checks
)

overall_pass_rate = (
    total_passed_checks
    / total_validation_checks
    * 100
)

overall_validation_summary = pd.DataFrame(
    {
        "Validation Property": [
            "Validation steps completed",
            "Total validation checks",
            "Passed validation checks",
            "Failed validation checks",
            "Overall pass rate",
            "Raw predictors validated",
            "Transformed features validated",
            "Application pages validated",
            "Prediction input methods validated",
            "Approved figures validated",
            "Invalid-input tests completed",
            "Final model",
            "Standard review cutoff",
            "Additional screening cutoff",
        ],
        "Validated Value": [
            6,
            total_validation_checks,
            total_passed_checks,
            total_failed_checks,
            f"{overall_pass_rate:.2f}%",
            43,
            179,
            8,
            3,
            len(approved_figures),
            len(step_4_input_validation_results),
            type(final_model).__name__,
            0.50,
            0.45,
        ],
    }
)

display(overall_validation_summary)

# 6. Confirm the complete validation suite passed
expected_step_check_counts = {
    1: 12,
    2: 18,
    3: 7,
    4: 18,
    5: 28,
    6: 15,
}

observed_step_check_counts = (
    final_validation_checks
    .groupby("Step Number")
    .size()
    .to_dict()
)

final_suite_checks = {
    "Six validation steps were completed": (
        len(final_step_summary) == 6
    ),
    "Expected validation-check counts are present": (
        observed_step_check_counts
        == expected_step_check_counts
    ),
    "All validation checks passed": (
        total_failed_checks == 0
    ),
    "Overall validation pass rate is 100%": (
        np.isclose(
            overall_pass_rate,
            100.0,
        )
    ),
    "All eight Streamlit pages were validated": (
        len(page_validation_table) == 8
        and (
            page_validation_table["Result"]
            == "PASSED"
        ).all()
    ),
    "All three input methods were validated": (
        len(input_method_table) == 3
        and (
            input_method_table["Result"]
            == "PASSED"
        ).all()
    ),
    "All 22 approved figures were validated": (
        len(approved_figure_table) == 22
        and (
            approved_figure_table["Result"]
            == "PASSED"
        ).all()
    ),
}

final_suite_validation_table = pd.DataFrame(
    [
        {
            "Final Validation Check": check_name,
            "Result": (
                "PASSED"
                if passed
                else "FAILED"
            ),
        }
        for check_name, passed
        in final_suite_checks.items()
    ]
)

display(final_suite_validation_table)


failed_final_suite_checks = [
    check_name
    for check_name, passed
    in final_suite_checks.items()
    if not passed
]

if failed_final_suite_checks:
    raise AssertionError(
        "Final application validation failed:\n"
        + "\n".join(
            f"- {check}"
            for check in failed_final_suite_checks
        )
    )

# 7. Create permanent output directories
VALIDATION_METRICS_DIRECTORY = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
)

VALIDATION_ARTIFACT_DIRECTORY = (
    PROJECT_ROOT
    / "artifacts"
)

VALIDATION_METRICS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

VALIDATION_ARTIFACT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

# 8. Define final saved-output paths
ALL_CHECKS_OUTPUT_PATH = (
    VALIDATION_METRICS_DIRECTORY
    / "notebook_9_streamlit_validation_checks.csv"
)

STEP_SUMMARY_OUTPUT_PATH = (
    VALIDATION_METRICS_DIRECTORY
    / "notebook_9_streamlit_validation_step_summary.csv"
)

OVERALL_SUMMARY_OUTPUT_PATH = (
    VALIDATION_METRICS_DIRECTORY
    / "notebook_9_streamlit_validation_overall_summary.csv"
)

PARITY_OUTPUT_PATH = (
    VALIDATION_METRICS_DIRECTORY
    / "notebook_9_direct_csv_parity_results.csv"
)

INPUT_TEST_OUTPUT_PATH = (
    VALIDATION_METRICS_DIRECTORY
    / "notebook_9_invalid_input_test_results.csv"
)

DOWNLOAD_TEST_OUTPUT_PATH = (
    VALIDATION_METRICS_DIRECTORY
    / "notebook_9_download_validation_results.csv"
)

FIGURE_TEST_OUTPUT_PATH = (
    VALIDATION_METRICS_DIRECTORY
    / "notebook_9_approved_figure_validation.csv"
)

JSON_SUMMARY_OUTPUT_PATH = (
    VALIDATION_ARTIFACT_DIRECTORY
    / "notebook_9_streamlit_validation_summary.json"
)

# 9. Save detailed validation CSV files
final_validation_checks.to_csv(
    ALL_CHECKS_OUTPUT_PATH,
    index=False,
)

final_step_summary.to_csv(
    STEP_SUMMARY_OUTPUT_PATH,
    index=False,
)

overall_validation_summary.to_csv(
    OVERALL_SUMMARY_OUTPUT_PATH,
    index=False,
)

step_3_direct_csv_parity_table.to_csv(
    PARITY_OUTPUT_PATH,
    index=False,
)

step_4_input_validation_results.to_csv(
    INPUT_TEST_OUTPUT_PATH,
    index=False,
)

step_5_validation_summary.to_csv(
    DOWNLOAD_TEST_OUTPUT_PATH,
    index=False,
)

approved_figure_table.to_csv(
    FIGURE_TEST_OUTPUT_PATH,
    index=False,
)

# 10. Create the machine-readable JSON summary
validation_timestamp = (
    datetime.now()
    .astimezone()
    .isoformat(timespec="seconds")
)

json_summary = {
    "notebook": (
        "09_streamlit_application_validation.ipynb"
    ),
    "validation_status": "PASSED",
    "validation_timestamp": validation_timestamp,
    "final_model": type(final_model).__name__,
    "prediction_target": input_schema[
        "prediction_target"
    ],
    "positive_class": int(
        input_schema["positive_class"]
    ),
    "raw_predictors": int(
        input_schema["feature_count"]
    ),
    "numeric_predictors": int(
        input_schema[
            "numeric_feature_count"
        ]
    ),
    "categorical_predictors": int(
        input_schema[
            "categorical_feature_count"
        ]
    ),
    "transformed_features": int(
        final_model.n_features_in_
    ),
    "standard_review_cutoff": float(
        input_schema["main_threshold"]
    ),
    "additional_screening_cutoff": float(
        input_schema[
            "recall_focused_threshold"
        ]
    ),
    "application_pages": 8,
    "prediction_input_methods": 3,
    "approved_figures": int(
        len(approved_figures)
    ),
    "invalid_input_tests": int(
        len(step_4_input_validation_results)
    ),
    "validation_steps": int(
        len(final_step_summary)
    ),
    "total_checks": total_validation_checks,
    "passed_checks": total_passed_checks,
    "failed_checks": total_failed_checks,
    "overall_pass_rate_percentage": round(
        overall_pass_rate,
        2,
    ),
    "step_summary": (
        json.loads(
            final_step_summary.to_json(
                orient="records"
            )
        )
    ),
    "saved_outputs": {
        "all_validation_checks": str(
            ALL_CHECKS_OUTPUT_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "step_summary": str(
            STEP_SUMMARY_OUTPUT_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "overall_summary": str(
            OVERALL_SUMMARY_OUTPUT_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "prediction_parity": str(
            PARITY_OUTPUT_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "invalid_input_results": str(
            INPUT_TEST_OUTPUT_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "download_validation": str(
            DOWNLOAD_TEST_OUTPUT_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
        "figure_validation": str(
            FIGURE_TEST_OUTPUT_PATH.relative_to(
                PROJECT_ROOT
            )
        ),
    },
}

with open(
    JSON_SUMMARY_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as json_file:
    json.dump(
        json_summary,
        json_file,
        indent=2,
    )

# 11. Confirm every saved output was created successfully
saved_validation_paths = {
    "All validation checks": (
        ALL_CHECKS_OUTPUT_PATH
    ),
    "Step-level summary": (
        STEP_SUMMARY_OUTPUT_PATH
    ),
    "Overall summary": (
        OVERALL_SUMMARY_OUTPUT_PATH
    ),
    "Prediction parity results": (
        PARITY_OUTPUT_PATH
    ),
    "Invalid-input results": (
        INPUT_TEST_OUTPUT_PATH
    ),
    "Download validation results": (
        DOWNLOAD_TEST_OUTPUT_PATH
    ),
    "Approved-figure validation": (
        FIGURE_TEST_OUTPUT_PATH
    ),
    "JSON validation summary": (
        JSON_SUMMARY_OUTPUT_PATH
    ),
}

saved_output_rows = []

for output_name, output_path in saved_validation_paths.items():
    output_exists = output_path.exists()
    output_size = (
        output_path.stat().st_size
        if output_exists
        else 0
    )

    saved_output_rows.append(
        {
            "Saved Output": output_name,
            "Relative Path": str(
                output_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "Exists": output_exists,
            "Size (bytes)": output_size,
            "Result": (
                "PASSED"
                if (
                    output_exists
                    and output_size > 0
                )
                else "FAILED"
            ),
        }
    )

saved_output_validation_table = pd.DataFrame(
    saved_output_rows
)

display(saved_output_validation_table)


saved_outputs_passed = (
    saved_output_validation_table["Result"]
    == "PASSED"
).all()

if not saved_outputs_passed:
    raise AssertionError(
        "One or more Notebook 09 validation files "
        "were not saved successfully."
    )

# 12. Preserve final notebook variables
notebook_9_final_validation_checks = (
    final_validation_checks.copy()
)

notebook_9_final_step_summary = (
    final_step_summary.copy()
)

notebook_9_overall_validation_summary = (
    overall_validation_summary.copy()
)

notebook_9_saved_output_validation = (
    saved_output_validation_table.copy()
)

# 13. Print final notebook conclusion
print("\n" + "=" * 78)
print("NOTEBOOK 09 FINAL RESULT: PASSED")
print("=" * 78)
print(f"Validation steps completed : {len(final_step_summary)}")
print(f"Total validation checks    : {total_validation_checks}")
print(f"Passed validation checks   : {total_passed_checks}")
print(f"Failed validation checks   : {total_failed_checks}")
print(f"Overall validation rate    : {overall_pass_rate:.2f}%")
print(f"Application pages          : 8")
print(f"Prediction input methods   : 3")
print(f"Approved figures available : {len(approved_figures)}")
print(f"Invalid-input tests passed : {len(step_4_input_validation_results)}")
print("-" * 78)
print(
    "The finalized Streamlit application passed all structural, "
    "prediction, explanation, input-validation, download, and "
    "deployment-asset checks."
)
print(
    "No model retraining, preprocessing changes, or threshold "
    "changes were performed."
)
print("=" * 78)